In [1]:
import pandas as pd

new_df = pd.read_csv('../data/processed_movies.csv')

In [2]:
new_df.head

<bound method NDFrame.head of           id                                     title  release_year  \
0      19995                                    Avatar          2009   
1        285  Pirates of the Caribbean: At World's End          2007   
2     206647                                   Spectre          2015   
3      49026                     The Dark Knight Rises          2012   
4      49529                               John Carter          2012   
...      ...                                       ...           ...   
4794    9367                               El Mariachi          1992   
4795   72766                                 Newlyweds          2011   
4796  231617                 Signed, Sealed, Delivered          2013   
4797  126186                          Shanghai Calling          2012   
4798   25975                         My Date with Drew          2005   

                                                   tags  
0     in the 22nd century a paraplegic marine i

In [3]:
new_df.shape

(4799, 4)

In [4]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Apply stemming to the tags

from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    return " ".join(
        [ps.stem(word) for word in text.split()]
    )

new_df['tags'] = new_df['tags'].apply(stem)

In [6]:
new_df['tags'].iloc[0]

'in the 22nd centuri a parapleg marin is dispatch to the moon pandora on a uniqu mission but becom torn between follow order and protect an alien civil action adventur fantasi sciencefict cultur clash futur space war space coloni societi space travel futurist romanc space alien tribe alien planet cgi marin soldier battl love affair anti war power relat mind and soul 3d samworthington zoesaldana sigourneyweav stephenlang michellerodriguez jamescameron millennium'

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()

In [8]:
vectors.shape

(4799, 5000)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors) # Similarity gives the similarity between every other movie

In [10]:
similarity.shape

(4799, 4799)

In [11]:
indices = pd.Series(new_df.index, index=new_df['title']).drop_duplicates()  # We don't need drop_dublicates because we have already removed it in the pre processing stage

In [12]:
def recommend(movie):

    # Get index of the movie
    index = indices[movie]

    # Similarity scores
    distances = similarity[index]

    # Sort movies by similarity
    movie_list = sorted(enumerate(distances), key=lambda x: x[1], reverse=True)

    # Remove itself and take top 5
    movie_list = movie_list[1:6]

    # Collect movie names
    recommended = []
    for i in movie_list:
        recommended.append(new_df.iloc[i[0]].title)

    return recommended

In [13]:
print(recommend("Batman Begins"))




['The Dark Knight', 'The Dark Knight Rises', 'Batman', 'Batman Returns', 'Batman & Robin']


In [14]:
new_df

,id,title,release_year,tags
0,19995,Avatar,2009,in the 22nd centuri a parapleg marin is dispat...
1,285,Pirates of the Caribbean: At World's End,2007,captain barbossa long believ to be dead ha com...
2,206647,Spectre,2015,a cryptic messag from bond past send him on a ...
3,49026,The Dark Knight Rises,2012,follow the death of district attorney harvey d...
4,49529,John Carter,2012,john carter is a warweari former militari capt...
...,...,...,...,...
4794,9367,El Mariachi,1992,el mariachi just want to play hi guitar and ca...
4795,72766,Newlyweds,2011,a newlyw coupl honeymoon is upend by the arriv...
4796,231617,"Signed, Sealed, Delivered",2013,sign seal deliv introduc a dedic quartet of ci...
4797,126186,Shanghai Calling,2012,when ambiti new york attorney sam is sent to s...


In [16]:
import pickle

pickle.dump(new_df, open('../movies.pkl', 'wb'))

In [18]:
pickle.dump(similarity, open('../similarity.pkl', 'wb'))